## Full implementation process of fine tuning flan-t5 model code ran on Google Colab.

In [ ]:
!pip install -q --no-cache-dir \
  "transformers==4.46.3" \
  "tokenizers==0.20.1" \
  "accelerate==0.34.2" \
  "peft==0.13.2" \
  sentencepiece safetensors

!pip install datasets evaluate rouge-score torch tensorboard

In [ ]:
import nltk
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
nltk.download("punkt", quiet=True)
nltk.download('punkt_tab')

# 1. Data Loading and Preparation

# Load BioLaySumm dataset
dataset = load_dataset("BioLaySumm/BioLaySumm2025-LaymanRRG-opensource-track")

# Dataset preparation clean up
def clean_up(example):
    src = (example["radiology_report"] or "").strip()
    tgt = (example["layman_report"] or "").strip()
    return len(src) > 0 and len(tgt) > 0

clean_dataset = dataset.filter(clean_up)

# Apply clean up on selected range of required datasets. 
subset_train = clean_dataset["train"].shuffle(seed=42).select(range(100000))
subset_val   = clean_dataset["validation"].select(range(8000))


2. Data Formatting and Preprocessing

# Model + tokenizer
MODEL_NAME = "google/flan-t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 4. Preprocessing 
PREFIX = "Summarize this radiology report for a layperson: "
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 128

def preprocess_function(batch):
    inputs = [PREFIX + x for x in batch["radiology_report"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="longest",          # pad dynamically, not to fixed length
    )

    labels = tokenizer(
        text_target=batch["layman_report"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="longest",
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

cols_to_keep = ["input_ids", "attention_mask", "labels"]

tokenized_train = subset_train.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in subset_train.column_names if c not in cols_to_keep],
)
tokenized_val = subset_val.map(
    preprocess_function,
    batched=True,
    remove_columns=[c for c in subset_val.column_names if c not in cols_to_keep],
)
print(first["labels"][:20])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # authorize access

In [ ]:
# 5. Metric
metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
   preds, labels = eval_preds

   # decode preds and labels
   labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
   decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
   decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

   # rougeLSum expects newline after each sentence
   decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
   decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

   result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

   return result

# 6. Training arguments
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/saved_models/",
    eval_strategy="epoch",
    learning_rate=2e-4, # smaller lr → more stable
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    num_train_epochs=3,
    logging_steps=10,
    predict_with_generate=True,
    push_to_hub=False,
    report_to="none",
)

model = model.to("cuda").float()

# 7. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 8. Train
trainer.train()
print(trainer.state.log_history[-10:])

In [ ]:
# 9. Save final model and tokenizer
SAVED_PATH = "/content/drive/MyDrive/saved_models/final"

final_dir = SAVED_PATH
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Final model and tokenizer saved to {final_dir}")